### 3.3 Attending to different parts of the input with self-attention

#### 3.3.1 A simple self-attention mechanism without trainable weights

In [1]:
import torch

In [2]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your (x^1)
     [0.55, 0.87, 0.66], # journey (x^2)
     [0.57, 0.85, 0.64], # starts (x^3)
     [0.22, 0.58, 0.33], # with (x^4)
     [0.77, 0.25, 0.10], # one (x^5)
     [0.05, 0.80, 0.55]] # step (x^6)
)

In [3]:
# attention scores
query = inputs[1]
attention_scores = torch.empty(len(inputs))

In [6]:

for i, x_i in enumerate(inputs):
    attention_scores[i] = torch.dot(x_i, query)

attention_scores

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

In [7]:
# Getting attention weights by the attention scores normalization

attn_weights = attention_scores / attention_scores.sum()

print(attn_weights)
attn_weights.sum()

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])


tensor(1.0000)

In [10]:
# using softmax instead nornalization

def softmax_naive(x: torch.tensor) -> torch.tensor:
    return torch.exp(x) / torch.exp(x).sum()

In [11]:
attn_weights_2 = softmax_naive(attention_scores)
print(attn_weights_2)
attn_weights_2.sum()

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


tensor(1.)

In [15]:
# torch softmax
attn_weights_3 = torch.softmax(attention_scores, dim=0)
print(attn_weights_3)
attn_weights_3.sum()

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


tensor(1.)

In [16]:
# Context vector = weighted sum of input vectors

context_1 = torch.zeros(len(query))
for i, x in enumerate(inputs):
    context_1 += x * attn_weights_3[i]

context_1

tensor([0.4419, 0.6515, 0.5683])

#### 3.3.2 Computing attention weights for all input tokens

In [17]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [22]:
attn_scores = inputs @ inputs.T 
attn_weights = torch.softmax(attn_scores, dim=-1)
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [24]:
attn_weights.sum(dim=1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [25]:
all_context_vecs = attn_weights @ inputs
all_context_vecs 

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [26]:
context_1

tensor([0.4419, 0.6515, 0.5683])

### 3.4 Implementing self-attention with trainable weights

#### 3.4.1 Computing the attention weights step by step

In [27]:
import torch

In [30]:
torch.manual_seed(123)

In [28]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your (x^1)
     [0.55, 0.87, 0.66], # journey (x^2)
     [0.57, 0.85, 0.64], # starts (x^3)
     [0.22, 0.58, 0.33], # with (x^4)
     [0.77, 0.25, 0.10], # one (x^5)
     [0.05, 0.80, 0.55]] # step (x^6)
)

In [29]:
x_2 = inputs[1]
d_in = len(x_2)
d_out = 2

d_in, d_out

(3, 2)

In [31]:
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [44]:
x_2, x_2.shape

(tensor([0.5500, 0.8700, 0.6600]), torch.Size([3]))

In [33]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

query_2.shape, key_2.shape, value_2.shape

(torch.Size([2]), torch.Size([2]), torch.Size([2]))

In [35]:
W_query

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])

In [34]:
query_2

tensor([0.4306, 1.4551])

In [53]:
queries = inputs @ W_query
queries

tensor([[0.2309, 1.0966],
        [0.4306, 1.4551],
        [0.4300, 1.4343],
        [0.2355, 0.7990],
        [0.2983, 0.6565],
        [0.2568, 1.0533]])

In [ ]:
keys = inputs @ W_key
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]])

In [39]:
query_2, key_2

(tensor([0.4306, 1.4551]), tensor([0.4433, 1.1419]))

In [41]:
# attn_score_2 = torch.dot(query_2, key_2)
attn_score_2 = query_2.dot(key_2)
attn_score_2

tensor(1.8524)

In [43]:
attn_scores_2 = query_2 @ keys.T
attn_scores_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])

In [54]:
attn_scores = queries @ keys.T
attn_scores

tensor([[0.9231, 1.3545, 1.3241, 0.7910, 0.4032, 1.1330],
        [1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
        [1.2544, 1.8284, 1.7877, 1.0654, 0.5508, 1.5238],
        [0.6973, 1.0167, 0.9941, 0.5925, 0.3061, 0.8475],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707, 0.7307],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]])

In [45]:
d_k = keys.shape[-1]
d_k

2

In [ ]:
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
attn_weights_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])

In [55]:
attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)
attn_weights

tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])

In [49]:
values = inputs @ W_value
values

tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]])

In [51]:
context_vec_2 = attn_weights_2 @ values
context_vec_2

tensor([0.3061, 0.8210])

In [56]:
context_vec = attn_weights @ values
context_vec

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]])

#### 3.4.2 Implementing a compact self-attention Python class

In [1]:
import torch
import torch.nn as nn

In [2]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=True)
        self.W_key = nn.Parameter(torch.rand(d_in, d_out), requires_grad=True)
        self.W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=True)

    def forward(self, x):
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value

        attn_scores = queries @ keys.T
        d_k = keys.shape[-1]  # d_out
        attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)
        context_vec = attn_weights @ values

        return context_vec

In [3]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your (x^1)
     [0.55, 0.87, 0.66], # journey (x^2)
     [0.57, 0.85, 0.64], # starts (x^3)
     [0.22, 0.58, 0.33], # with (x^4)
     [0.77, 0.25, 0.10], # one (x^5)
     [0.05, 0.80, 0.55]] # step (x^6)
)

In [4]:
d_in = inputs.shape[-1]
d_out = 2

d_in, d_out

(3, 2)

In [5]:
# Context vectors as the output of `SelfAttention_v1` layer
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)

sa_v1(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [6]:
# Using `Linear` layer instead nn.Parameters

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        d_k = keys.shape[-1]  # d_out
        attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)
        context_vec = attn_weights @ values

        return context_vec

In [ ]:
# Context vectors as the output of `SelfAttention_v2` layer
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)

sa_v2(inputs)

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)

#### Exercise 3.1 Comparing SelfAttention_v1 and SelfAttention_v2

In [21]:
print(sa_v2)

SelfAttention_v2(
  (W_query): Linear(in_features=3, out_features=2, bias=False)
  (W_key): Linear(in_features=3, out_features=2, bias=False)
  (W_value): Linear(in_features=3, out_features=2, bias=False)
)


In [29]:
sa_v2.W_query.weight

Parameter containing:
tensor([[-0.2354,  0.0191, -0.2867],
        [ 0.2177, -0.4919,  0.4232]], requires_grad=True)

In [13]:
next(sa_v2.parameters())

Parameter containing:
tensor([[-0.2354,  0.0191, -0.2867],
        [ 0.2177, -0.4919,  0.4232]], requires_grad=True)

In [16]:
sa_v1.W_query

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]], requires_grad=True)

In [18]:
sa_v1.W_query = nn.Parameter(sa_v2.W_query.weight.T)
sa_v1.W_key = nn.Parameter(sa_v2.W_key.weight.T)
sa_v1.W_value = nn.Parameter(sa_v2.W_value.weight.T)

In [19]:
sa_v1(inputs)

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)

In [20]:
sa_v2(inputs)

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)

### 3.5 Hiding future words with causal attention

#### 3.5.1 Applying a causal attention mask

In [1]:
import torch
import torch.nn as nn

In [2]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your (x^1)
     [0.55, 0.87, 0.66], # journey (x^2)
     [0.57, 0.85, 0.64], # starts (x^3)
     [0.22, 0.58, 0.33], # with (x^4)
     [0.77, 0.25, 0.10], # one (x^5)
     [0.05, 0.80, 0.55]] # step (x^6)
)

In [3]:
d_in = inputs.shape[-1]
d_out = 2

d_in, d_out

(3, 2)

In [4]:
# Using `Linear` layer instead nn.Parameters

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        d_k = keys.shape[-1]  # d_out
        attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)
        context_vec = attn_weights @ values

        return context_vec

In [ ]:
# Context vectors as the output of `SelfAttention_v2` layer
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)

In [8]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / d_out**0.5, dim=-1)

attn_weights

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)

In [9]:
context_len = len(attn_scores)
simple_mask = torch.tril(torch.ones(context_len, context_len))

simple_mask

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [10]:
masked_simple = attn_weights * simple_mask
masked_simple

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)

In [12]:
masked_simple.sum(dim=-1, keepdim=True)

tensor([[0.1921],
        [0.3700],
        [0.5357],
        [0.6775],
        [0.8415],
        [1.0000]], grad_fn=<SumBackward1>)

In [13]:
masked_simple_norm = masked_simple / masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)

In [15]:
mask = torch.triu(torch.ones(context_len, context_len), diagonal=1)
mask.bool()

tensor([[False,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True],
        [False, False, False, False,  True,  True],
        [False, False, False, False, False,  True],
        [False, False, False, False, False, False]])

In [19]:
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
masked

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)

In [20]:
attn_weights = torch.softmax(masked / d_out**0.5, dim=-1)
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)

#### 3.5.2 Masking additional attention weights with dropout

In [21]:
torch.manual_seed(123)

dropout = nn.Dropout(0.5)
ones = torch.ones(6,6)
dropout(ones)

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])

In [26]:
print(dropout)

Dropout(p=0.5, inplace=False)


In [28]:
torch.manual_seed(123)
dropout(attn_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)

#### 3.5.3 Implementing a compact causal attention class

In [30]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your (x^1)
     [0.55, 0.87, 0.66], # journey (x^2)
     [0.57, 0.85, 0.64], # starts (x^3)
     [0.22, 0.58, 0.33], # with (x^4)
     [0.77, 0.25, 0.10], # one (x^5)
     [0.05, 0.80, 0.55]] # step (x^6)
)

In [37]:
d_in = inputs.shape[1]
d_out = 2

d_in, d_out

(3, 2)

In [33]:
batch = torch.stack((inputs,inputs), dim=0)

print(batch.shape)
batch

torch.Size([2, 6, 3])


tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])

In [34]:
batch[0]

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [ ]:
class CausalAttention(nn.Module):
    
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
            )
        
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1,2)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.d_out**0.5, dim=-1)

        context_vecs = self.dropout(attn_weights) @ values

        return context_vecs

In [38]:
context_length = batch.shape[1]
context_length, d_in, d_out

(6, 3, 2)

In [39]:
torch.manual_seed(123)
ca = CausalAttention(d_in, d_out, context_length, dropout=0.0)
context_vec = ca(batch)

context_vec.shape

torch.Size([2, 6, 2])

In [49]:
ca.W_query(batch)


tensor([[[-0.3536,  0.3965],
         [-0.3021, -0.0289],
         [-0.3015, -0.0232],
         [-0.1353, -0.0978],
         [-0.2052,  0.0870],
         [-0.1542, -0.1499]],

        [[-0.3536,  0.3965],
         [-0.3021, -0.0289],
         [-0.3015, -0.0232],
         [-0.1353, -0.0978],
         [-0.2052,  0.0870],
         [-0.1542, -0.1499]]], grad_fn=<UnsafeViewBackward0>)

In [46]:
ca.W_key(batch).transpose(1,2)

tensor([[[-0.5740, -0.8709, -0.8628, -0.4789, -0.4744, -0.5888],
         [ 0.2727,  0.1008,  0.1060,  0.0051,  0.1696, -0.0388]],

        [[-0.5740, -0.8709, -0.8628, -0.4789, -0.4744, -0.5888],
         [ 0.2727,  0.1008,  0.1060,  0.0051,  0.1696, -0.0388]]],
       grad_fn=<TransposeBackward0>)

In [54]:
atn_sc = ca.W_query(batch) @ ca.W_key(batch).transpose(1,2)
print(atn_sc.shape)
atn_sc

torch.Size([2, 6, 6])


tensor([[[0.3111, 0.3479, 0.3471, 0.1714, 0.2350, 0.1928],
         [0.1655, 0.2602, 0.2576, 0.1445, 0.1384, 0.1790],
         [0.1667, 0.2602, 0.2577, 0.1443, 0.1391, 0.1784],
         [0.0510, 0.1080, 0.1064, 0.0643, 0.0476, 0.0835],
         [0.1415, 0.1875, 0.1863, 0.0987, 0.1121, 0.1174],
         [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],

        [[0.3111, 0.3479, 0.3471, 0.1714, 0.2350, 0.1928],
         [0.1655, 0.2602, 0.2576, 0.1445, 0.1384, 0.1790],
         [0.1667, 0.2602, 0.2577, 0.1443, 0.1391, 0.1784],
         [0.0510, 0.1080, 0.1064, 0.0643, 0.0476, 0.0835],
         [0.1415, 0.1875, 0.1863, 0.0987, 0.1121, 0.1174],
         [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]]],
       grad_fn=<UnsafeViewBackward0>)

In [55]:
print(ca.mask.bool().shape)
ca.mask.bool()

torch.Size([6, 6])


tensor([[False,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True],
        [False, False, False, False,  True,  True],
        [False, False, False, False, False,  True],
        [False, False, False, False, False, False]])

In [56]:
atn_sc.masked_fill_(ca.mask.bool(), -torch.inf)

atn_sc

tensor([[[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
         [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
         [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
         [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
         [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
         [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],

        [[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
         [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
         [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
         [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
         [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
         [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]]],
       grad_fn=<MaskedFillBackward0>)

In [59]:
atn_w = torch.softmax(atn_sc / d_out**0.5, dim=-1)
print(atn_w.shape)
atn_w

torch.Size([2, 6, 6])


tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
         [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
         [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
         [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
         [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
         [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
         [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]]],
       grad_fn=<SoftmaxBackward0>)

In [61]:
print(ca.dropout(atn_w).shape)
print(ca.W_value(batch).shape)

torch.Size([2, 6, 6])
torch.Size([2, 6, 2])


In [62]:
ca.W_value(batch)

tensor([[[-0.4519,  0.2216],
         [-0.7142, -0.1961],
         [-0.7127, -0.1971],
         [-0.3809, -0.1557],
         [-0.4861, -0.1597],
         [-0.4213, -0.1501]],

        [[-0.4519,  0.2216],
         [-0.7142, -0.1961],
         [-0.7127, -0.1971],
         [-0.3809, -0.1557],
         [-0.4861, -0.1597],
         [-0.4213, -0.1501]]], grad_fn=<UnsafeViewBackward0>)

In [60]:
cntx = ca.dropout(atn_w) @ ca.W_value(batch)
print(cntx.shape)
cntx


torch.Size([2, 6, 2])


tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)

### 3.6 Extending single-head attention to multi-head attention

#### 3.6.1 Stacking multiple single-head attention layers

In [1]:
import torch
import torch.nn as nn

In [2]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your (x^1)
     [0.55, 0.87, 0.66], # journey (x^2)
     [0.57, 0.85, 0.64], # starts (x^3)
     [0.22, 0.58, 0.33], # with (x^4)
     [0.77, 0.25, 0.10], # one (x^5)
     [0.05, 0.80, 0.55]] # step (x^6)
)

In [3]:
class CausalAttention(nn.Module):
    
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
            )
        
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1,2)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.d_out**0.5, dim=-1)

        context_vecs = self.dropout(attn_weights) @ values

        return context_vecs

In [9]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self,
                 d_in,
                 d_out,
                 context_length,
                 dropout,
                 num_heads,qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
                for _ in range(num_heads)]
                )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


In [4]:
d_in = inputs.shape[1]
d_out = 2

d_in, d_out

(3, 2)

In [5]:
batch = torch.stack((inputs,inputs), dim=0)

print(batch.shape)
batch

torch.Size([2, 6, 3])


tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])

In [7]:
torch.manual_seed(123)
context_length = batch.shape[1] # This is the number of tokens

context_length

6

In [10]:
mha = MultiHeadAttentionWrapper( d_in, d_out, context_length, 0.0, num_heads=2)
context_vec = mha(batch)

print(context_vec.shape)
context_vec

torch.Size([2, 6, 4])


tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)

#### Exercise 3.2 Returning two-dimensional embedding vectors

In [13]:
d_out_e=1
mha1 = MultiHeadAttentionWrapper(d_in, d_out_e, context_length, 0.0, num_heads=2)
context_vec = mha1(batch)

print(context_vec.shape)
context_vec

torch.Size([2, 6, 2])


tensor([[[-0.3749,  0.5063],
         [-0.1520,  0.6623],
         [-0.0699,  0.7112],
         [-0.0191,  0.6541],
         [ 0.0127,  0.6026],
         [ 0.0186,  0.6018]],

        [[-0.3749,  0.5063],
         [-0.1520,  0.6623],
         [-0.0699,  0.7112],
         [-0.0191,  0.6541],
         [ 0.0127,  0.6026],
         [ 0.0186,  0.6018]]], grad_fn=<CatBackward0>)

#### 3.6.2 Implementing multi-head attention with weight splits

In [ ]:
a = torch.tensor(
    [[[[0.2745, 0.6584, 0.2775, 0.8573],
       [0.8993, 0.0390, 0.9268, 0.7388],
       [0.7179, 0.7058, 0.9156, 0.4340]],
       [[0.0772, 0.3565, 0.1479, 0.5331],
       [0.4066, 0.2318, 0.4545, 0.9737],
       [0.4606, 0.5159, 0.4220, 0.5786]]]])

a.shape
#(b, num_heads, num_tokens, head_dim)

torch.Size([1, 2, 3, 4])

In [15]:
at = a.transpose(2,3)
print(at.shape)
at

torch.Size([1, 2, 4, 3])


tensor([[[[0.2745, 0.8993, 0.7179],
          [0.6584, 0.0390, 0.7058],
          [0.2775, 0.9268, 0.9156],
          [0.8573, 0.7388, 0.4340]],

         [[0.0772, 0.4066, 0.4606],
          [0.3565, 0.2318, 0.5159],
          [0.1479, 0.4545, 0.4220],
          [0.5331, 0.9737, 0.5786]]]])

In [16]:
r = a @ at
print(r.shape)
r

torch.Size([1, 2, 3, 3])


tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])

In [18]:
first_head = a[0, 0, :, :]
first_head

tensor([[0.2745, 0.6584, 0.2775, 0.8573],
        [0.8993, 0.0390, 0.9268, 0.7388],
        [0.7179, 0.7058, 0.9156, 0.4340]])

In [ ]:
first_res = first_head @ first_head.T
first_res # the result is the same as `r`

tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

In [21]:
second_head = a[0, 1, :, :]
second_head

tensor([[0.0772, 0.3565, 0.1479, 0.5331],
        [0.4066, 0.2318, 0.4545, 0.9737],
        [0.4606, 0.5159, 0.4220, 0.5786]])

In [ ]:
second_head @ second_head.T   # the result is the same as `r`

tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])

---

In [36]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out,
                 context_length,
                 dropout,
                 num_heads,
                 qkv_bias=False):
        
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)  # -> (b, num_tokens, d_out)
        querys = self.W_query(x)  # -> (b, num_tokens, d_out)
        values = self.W_value(x)  # -> (b, num_tokens, d_out)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim ) # -> (b, num_tokens, num_heads, head_dim)
        querys = querys.view(b, num_tokens, self.num_heads, self.head_dim ) # -> (b, num_tokens, num_heads, head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim ) # -> (b, num_tokens, num_heads, head_dim)

        keys = keys.transpose(1,2)    # -> (b, num_heads, num_tokens, head_dim)
        querys = querys.transpose(1,2)   # -> (b, num_heads, num_tokens, head_dim)
        values = values.transpose(1,2)   # -> (b, num_heads, num_tokens, head_dim)

        attn_scores_ = querys @ keys.transpose(2,3)  # -> (b, num_heads, num_tokens, num_tokens)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores_.masked_fill_(mask_bool, -torch.inf)

        attn_weights = self.dropout(
            torch.softmax(attn_scores_ / keys.shape[-1]**0.5, dim=-1))  # -> (b, num_heads, num_tokens, num_tokens)

        context_vecs = attn_weights @ values  # -> (b, num_heads, num_tokens, head_dim)
        context_vecs = context_vecs.transpose(1,2)   # -> (b, num_tokens, num_heads, head_dim)
        context_vecs = context_vecs.contiguous().view(b, num_tokens, self.d_out)   # -> (b, num_tokens, d_out)
        context_vecs = self.out_proj(context_vecs)   # -> (b, num_tokens, d_out)

        return context_vecs   # -> (b, num_tokens, d_out)


In [37]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2

batch_size, context_length, d_in, d_out

(2, 6, 3, 2)

In [38]:
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)

print(context_vecs.shape)
context_vecs

torch.Size([2, 6, 2])


tensor([[[ 0.1257, -0.1968],
         [ 0.1007, -0.2878],
         [ 0.0920, -0.3188],
         [ 0.0762, -0.2947],
         [ 0.0710, -0.2882],
         [ 0.0644, -0.2800]],

        [[ 0.1257, -0.1968],
         [ 0.1007, -0.2878],
         [ 0.0920, -0.3188],
         [ 0.0762, -0.2947],
         [ 0.0710, -0.2882],
         [ 0.0644, -0.2800]]], grad_fn=<UnsafeViewBackward0>)

#### Exercise 3.3 Initializing GPT-2 size attention modules

In [39]:
d_in = d_out = 768
context_length = 1024
num_heads = 12

mha_gpt2 = MultiHeadAttention(
    d_in=d_in,
    d_out=d_out,
    context_length=context_length,
    dropout=0.0,
    num_heads=num_heads)

In [40]:
inputs = torch.rand(2,6,d_in)

print(inputs.shape)
inputs

torch.Size([2, 6, 768])


tensor([[[0.7023, 0.5590, 0.1891,  ..., 0.5121, 0.7304, 0.8090],
         [0.5002, 0.0511, 0.2869,  ..., 0.3850, 0.0151, 0.0572],
         [0.2848, 0.5213, 0.4476,  ..., 0.3986, 0.5123, 0.6619],
         [0.8044, 0.4216, 0.7890,  ..., 0.6279, 0.0371, 0.9637],
         [0.8582, 0.5726, 0.5360,  ..., 0.4768, 0.4318, 0.8418],
         [0.5061, 0.7434, 0.8083,  ..., 0.1494, 0.1254, 0.5796]],

        [[0.6136, 0.0523, 0.8029,  ..., 0.8863, 0.8455, 0.2252],
         [0.4650, 0.2896, 0.5469,  ..., 0.0518, 0.0657, 0.7637],
         [0.5025, 0.9672, 0.0052,  ..., 0.5751, 0.9195, 0.3735],
         [0.9358, 0.1336, 0.6062,  ..., 0.7767, 0.5125, 0.2941],
         [0.0272, 0.7237, 0.5436,  ..., 0.0520, 0.3642, 0.4925],
         [0.3497, 0.4188, 0.8730,  ..., 0.2451, 0.4716, 0.8973]]])

In [41]:
con_vec = mha_gpt2(inputs)

print(con_vec.shape)
con_vec

torch.Size([2, 6, 768])


tensor([[[ 0.1280,  0.1323, -0.6128,  ...,  0.3136, -0.1051,  0.1590],
         [ 0.0255,  0.1570, -0.4744,  ...,  0.1735,  0.0140,  0.1831],
         [ 0.0225,  0.2137, -0.4553,  ...,  0.1972,  0.0153,  0.1581],
         [ 0.0498,  0.2258, -0.4476,  ...,  0.2024, -0.0043,  0.1323],
         [ 0.0171,  0.2536, -0.4576,  ...,  0.1942,  0.0012,  0.1420],
         [ 0.0042,  0.2370, -0.4213,  ...,  0.2029, -0.0147,  0.1592]],

        [[-0.0152,  0.2557, -0.3821,  ...,  0.1806,  0.0691,  0.1946],
         [ 0.0672,  0.2578, -0.4213,  ...,  0.2065,  0.0331,  0.2621],
         [ 0.0540,  0.2124, -0.4205,  ...,  0.2192,  0.0057,  0.2369],
         [ 0.0458,  0.2523, -0.3964,  ...,  0.2160,  0.0203,  0.1802],
         [ 0.0394,  0.2366, -0.3990,  ...,  0.1763,  0.0328,  0.2124],
         [ 0.0293,  0.2276, -0.3931,  ...,  0.1812,  0.0285,  0.1974]]],
       grad_fn=<UnsafeViewBackward0>)

In [42]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [43]:
count_parameters(mha_gpt2)

2359296